In [ ]:
import os
import random
from pathlib import Path
import numpy as np
import pandas as pd
import optuna
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor

SEED=13
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
P=Path('estonia_day_ahead_data.csv')
D=pd.read_csv(P)
T=pd.to_datetime(D['datetime_utc'],utc=True).dt.tz_convert('Europe/Tallinn').dt.tz_localize(None)
S=pd.DataFrame({'actual':pd.to_numeric(D['Estonia_actual'],errors='coerce').to_numpy(),'official':pd.to_numeric(D['Estonia_forecast'],errors='coerce').to_numpy()},index=T).sort_index()
S=S[~S.index.duplicated(keep='first')].asfreq('h').ffill()
X=[]
Y=[]
O=[]
Q=[]
for q in pd.date_range('2023-01-01','2025-12-31',freq='D'):
    if q not in S.index: continue
    i=S.index.get_loc(q)
    if not isinstance(i,(int,np.integer)) or i<168 or i+24>len(S): continue
    x=S['actual'].iloc[i-168:i].to_numpy(float)
    y=S['actual'].iloc[i:i+24].to_numpy(float)
    o=S['official'].iloc[i:i+24].to_numpy(float)
    if np.isfinite(x).all() and np.isfinite(y).all():
        X.append(x);Y.append(y);O.append(o);Q.append(q)
X=np.asarray(X,dtype=np.float32)
Y=np.asarray(Y,dtype=np.float32)
O=np.asarray(O,dtype=np.float32)
Q=pd.DatetimeIndex(Q)
dow=Q.dayofweek.to_numpy()
doy=Q.dayofyear.to_numpy()
mon=Q.month.to_numpy()
E=np.c_[np.sin(2*np.pi*dow/7),np.cos(2*np.pi*dow/7),np.sin(2*np.pi*(doy-1)/365.25),np.cos(2*np.pi*(doy-1)/365.25),np.sin(2*np.pi*(mon-1)/12),np.cos(2*np.pi*(mon-1)/12),(dow>=5).astype(float)].astype(np.float32)
tr=Q.year<2025
te=Q.year==2025
X0=X[tr];X1=X[te];E0=E[tr];E1=E[te];Y0=Y[tr];Y1=Y[te];O1=O[te];Q1=Q[te]
A0=np.c_[X0,E0];A1=np.c_[X1,E1]

def score(y,p):
    z=p-y
    return {'RMSE':float(np.sqrt(np.mean(z*z))),'MAE':float(np.mean(np.abs(z))),'Bias':float(np.mean(z)),'1-MAPE':float((1-np.mean(np.abs(z/y)))*100)}

def fit_xgb(a,b,p):
    m=XGBRegressor(objective='reg:squarederror',random_state=SEED,n_jobs=-1,tree_method='hist',**p)
    m.fit(a,b)
    return m

def tpe(a,b):
    k=len(a)//2
    at,av=a[:k],a[k:]
    bt,bv=b[:k],b[k:]
    def f(t):
        p={'n_estimators':t.suggest_int('n_estimators',200,1000),'learning_rate':t.suggest_float('learning_rate',0.01,0.15,log=True),'max_depth':t.suggest_int('max_depth',3,10),'min_child_weight':t.suggest_float('min_child_weight',1,10),'subsample':t.suggest_float('subsample',0.6,1.0),'colsample_bytree':t.suggest_float('colsample_bytree',0.6,1.0),'gamma':t.suggest_float('gamma',0,5),'reg_alpha':t.suggest_float('reg_alpha',1e-8,10,log=True),'reg_lambda':t.suggest_float('reg_lambda',1e-3,20,log=True)}
        m=fit_xgb(at,bt,p)
        return float(np.sqrt(mean_squared_error(bv,m.predict(av))))
    s=optuna.create_study(direction='minimize',sampler=optuna.samplers.TPESampler(seed=SEED,n_startup_trials=10,n_ei_candidates=24,prior_weight=1.0))
    s.optimize(f,n_trials=20)
    return s

R={}
PRED={}
PRED['Persistence']=X1[:,-24:].copy()
R['Persistence']=score(Y1,PRED['Persistence'])
PRED['Official']=O1.copy()
R['Official']=score(Y1,PRED['Official'])
xs=StandardScaler().fit(X0)
ys=StandardScaler().fit(Y0)
xt=xs.transform(X0);xe=xs.transform(X1);yt=ys.transform(Y0)
r=Ridge(alpha=1.0).fit(xt,yt)
PRED['Ridge']=ys.inverse_transform(r.predict(xe));R['Ridge']=score(Y1,PRED['Ridge'])
s=MultiOutputRegressor(SVR(kernel='rbf',C=10.0,epsilon=0.1,gamma='scale'),n_jobs=-1).fit(xt,yt)
PRED['SVR']=ys.inverse_transform(s.predict(xe));R['SVR']=score(Y1,PRED['SVR'])
f=RandomForestRegressor(n_estimators=300,max_depth=None,min_samples_leaf=1,random_state=SEED,n_jobs=-1).fit(X0,Y0)
PRED['Random Forest']=f.predict(X1);R['Random Forest']=score(Y1,PRED['Random Forest'])
v={'n_estimators':500,'learning_rate':0.03,'max_depth':6,'min_child_weight':1,'subsample':0.8,'colsample_bytree':0.8,'gamma':0.0,'reg_alpha':0.0,'reg_lambda':1.0}
x=fit_xgb(X0,Y0,v);PRED['XGBoost']=x.predict(X1);R['XGBoost']=score(Y1,PRED['XGBoost'])
se=tpe(X0,Y0);x=fit_xgb(X0,Y0,se.best_params);PRED['TPE-XGB']=x.predict(X1);R['TPE-XGB']=score(Y1,PRED['TPE-XGB'])
xa=fit_xgb(A0,Y0,v);PRED['XGBoost+ERA']=xa.predict(A1);R['XGBoost+ERA']=score(Y1,PRED['XGBoost+ERA'])
sea=tpe(A0,Y0);xa=fit_xgb(A0,Y0,sea.best_params);PRED['TPE-XGB+ERA']=xa.predict(A1);R['TPE-XGB+ERA']=score(Y1,PRED['TPE-XGB+ERA'])

class MLP(nn.Module):
    def __init__(self,n):
        super().__init__();self.m=nn.Sequential(nn.Linear(n,256),nn.ReLU(),nn.Dropout(.1),nn.Linear(256,128),nn.ReLU(),nn.Dropout(.1),nn.Linear(128,24))
    def forward(self,x): return self.m(x)

class LSTM(nn.Module):
    def __init__(self):
        super().__init__();self.r=nn.LSTM(1,128,2,batch_first=True,dropout=.1);self.o=nn.Linear(128,24)
    def forward(self,x): return self.o(self.r(x.unsqueeze(-1))[0][:,-1])

class TCN(nn.Module):
    def __init__(self):
        super().__init__();self.c=nn.Sequential(nn.Conv1d(1,32,3,padding=2,dilation=1),nn.ReLU(),nn.Dropout(.1),nn.Conv1d(32,64,3,padding=4,dilation=2),nn.ReLU(),nn.Dropout(.1),nn.Conv1d(64,128,3,padding=8,dilation=4),nn.ReLU(),nn.Dropout(.1));self.o=nn.Linear(128,24)
    def forward(self,x): return self.o(self.c(x.unsqueeze(1))[:,:,-1])

def torchfit(m,a,b,c,epochs=150):
    sx=StandardScaler().fit(a);sy=StandardScaler().fit(b)
    aa=sx.transform(a).astype(np.float32);bb=sy.transform(b).astype(np.float32);cc=sx.transform(c).astype(np.float32)
    dl=DataLoader(TensorDataset(torch.from_numpy(aa),torch.from_numpy(bb)),batch_size=64,shuffle=True,generator=torch.Generator().manual_seed(SEED))
    m=m.to(DEVICE);op=torch.optim.Adam(m.parameters(),lr=1e-3);ls=nn.MSELoss()
    for e in range(epochs):
        m.train()
        for u,w in dl:
            u=u.to(DEVICE);w=w.to(DEVICE);op.zero_grad();z=ls(m(u),w);z.backward();op.step()
    m.eval()
    with torch.no_grad(): p=m(torch.from_numpy(cc).to(DEVICE)).cpu().numpy()
    return sy.inverse_transform(p)

for n,m in [('MLP',MLP(168)),('LSTM',LSTM()),('TCN',TCN())]:
    PRED[n]=torchfit(m,X0,Y0,X1);R[n]=score(Y1,PRED[n])
for n,m in [('MLP+ERA',MLP(175))]:
    PRED[n]=torchfit(m,A0,Y0,A1);R[n]=score(Y1,PRED[n])
print(pd.DataFrame(R).T.sort_values('RMSE'))
print('TPE-XGB',se.best_params)
print('TPE-XGB+ERA',sea.best_params)